# Module 2: The model interface (notebook prototype)

> **Prototype.** This notebook tries out running module 2's observation steps here instead of in a
> terminal: every run's evidence stays on the page, in one table. The wording is placeholder; the
> module's prose belongs to the lab's author.

Run the cells from top to bottom. Each run of the agent is the same command as in the terminal, and
the evidence table below it reads what that run saved under `agent/runs/`.

In [ ]:
%cd ../agent
import json, pathlib, tomllib
from urllib.parse import urlparse
from IPython.display import Markdown, display

RUNS = pathlib.Path("runs")
CONFIG = tomllib.loads(pathlib.Path("config.toml").read_text())
PROMPTS = {CONFIG["agent"]["system_prompt"]: "default", CONFIG["agent"]["system_prompt_strong"]: "strong"}


def _call_01(run):
    request = json.loads((run / "01-request.json").read_text())
    response_path = run / "01-response.json"
    response = json.loads(response_path.read_text()) if response_path.exists() else {}
    return request, (response.get("choices") or [{}])[0]


def evidence(last=6):
    """One row per agent run, newest last: what varied, and what came back on model call 01."""
    runs = sorted(r for r in RUNS.glob("*") if (r / "summary.json").exists())[-last:]
    if not runs:
        return display(Markdown("No runs yet."))
    rows = ["| run | endpoint | prompt | finish_reason | tool_calls | turns |", "|---|---|---|---|---|---|"]
    for run in runs:
        summary = json.loads((run / "summary.json").read_text())
        request, choice = _call_01(run)
        calls = (choice.get("message") or {}).get("tool_calls")
        shown = ", ".join(c["function"]["name"] for c in calls) if calls else json.dumps(calls)
        endpoint = (urlparse(summary["endpoint"]).hostname or "").split(".")[0].removesuffix("-predictor")
        prompt = PROMPTS.get(request["messages"][0]["content"], "other")
        rows.append(f"| {run.name} | {endpoint} | {prompt} | {choice.get('finish_reason')} | `{shown}` | {summary['turns']} |")
    display(Markdown("\n".join(rows)))


def latest_content():
    """Model call 01's content in the newest run, exactly as the endpoint returned it."""
    runs = sorted(r for r in RUNS.glob("*") if (r / "summary.json").exists())
    if not runs:
        return display(Markdown("No runs yet."))
    _, choice = _call_01(runs[-1])
    content = (choice.get("message") or {}).get("content")
    shown = "null" if content is None else content
    display(Markdown(f"Run `{runs[-1].name}`, model call 01, `content`:\n\n```text\n{shown}\n```"))

## 1. Module 1's failure, again

In [ ]:
!python agent.py --q m1
evidence()

## 2. A stronger prompt

Every request the agent sends starts with a **system prompt**: instructions the model reads before
the question. The agent's is `system_prompt` in `config.toml`, and as the lab ships it, it says:

> You are a field service parts assistant for Kestrel Industrial, a manufacturer of industrial
> process chillers (K-200, K-400, K-600). Help service engineers find the right part and check
> whether they can get it. Be concise.

The **stronger prompt** is `system_prompt_strong`: the same text with three sentences added that
tell the model, as firmly as a prompt can, to use its tools:

> You MUST use the available tools to look up part numbers, stock and other facts. Never answer from
> memory. Always call a tool first.

`python agent.py --strong-prompt` sends that version instead. Nothing else in the request changes:
the same question, the same tools, the same endpoint. Making the instructions more forceful is the
first fix most people reach for when an agent won't use its tools.

The next cell shows the added sentences as they are in your `config.toml`, and the one after runs the
agent with the stronger prompt.

In [ ]:
default, strong = CONFIG["agent"]["system_prompt"], CONFIG["agent"]["system_prompt_strong"]
added = strong[len(default):].strip() if strong.startswith(default) else strong
display(Markdown(f"**Added in the stronger prompt:**\n\n> {added}"))

In [ ]:
!python agent.py --q m1 --strong-prompt
evidence()

**Your read:** did anything in the table change between the two rows? What does that tell you about
where the problem is?

## 3. What did the model actually return?

In [ ]:
latest_content()

The model did make the call: it's in the text above, as `<tool_call>{...}</tool_call>`. It never
reached `tool_calls`, and nothing reported an error.

**Your read:** the prompt isn't the problem, and the model did its part. What sits between the
model and your agent that should have turned that text into `tool_calls`?

## 4. Predict

The lab runs a second endpoint, `qwen3-8b-b`: the same model with the same weights. If the
identical request went to it, what would you expect `finish_reason`, `tool_calls` and `content` to
hold? Keep your guess in mind; the next cell sends exactly that request.

## 5. One identical request to both endpoints

In [ ]:
!COLUMNS=110 python compare_endpoints.py

Same model, same weights, same request. Only the endpoint differs, so the difference is in how each
endpoint is served: its **tool-call parser**, the serving setting that reads the model's text and
turns a tool call into `tool_calls`. On endpoint A it doesn't match this model's output, so it finds
nothing and returns the text as an ordinary answer, with no error.

Optional: confirm it in the configuration. In the OpenShift console, project `llm-serving`, open the
ServingRuntimes `qwen3-8b-a` and `qwen3-8b-b` and compare their `--tool-call-parser` arguments.

## 6. The same evidence in MLflow

Open the OpenShift AI dashboard, then **MLflow**, the `kestrel-agent` experiment, and **Traces**. Each
run above is a trace, tagged with its `endpoint`, `prompt` and `question`. Add those tags as columns to
see the table above in MLflow. Open a trace and select `model_call_1`: its inputs hold the full
messages (including the system prompt) and its outputs hold `finish_reason`, `tool_calls` and
`content`.

## 7. Repoint

Open `agent/config.toml` in the editor, change `qwen3-8b-a-predictor` to `qwen3-8b-b-predictor` on
the `base_url` line, and save. This cell confirms what the agent will call:

In [ ]:
print(tomllib.loads(pathlib.Path("config.toml").read_text())["model"]["base_url"])

## 8. Run it on the validated endpoint

In [ ]:
!python agent.py --q m1
evidence()

Look at the last row: `finish_reason` and `tool_calls` on model call 01, and the number of turns. If
the run used several turns, each calling `find_part` again, that's module 3's failure arriving early.